# Deliverable 2 — Pokémon Blue con un modelo de 1.5B

Este notebook reproduce **todo** lo que muestra el video y el documento técnico:

1. corre los tests de lógica (sin GPU),
2. carga `Qwen/Qwen2.5-1.5B-Instruct`,
3. evalúa el baseline, tres alternativas y la solución sobre **los mismos 50 estados** (≈15 min en una T4),
4. muestra la demo: un estado **nuevo**, con semilla sacada del reloj, baseline y solución lado a lado,
5. repite un caso donde la solución falla.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

In [ ]:
!nvidia-smi -L

## 1. Código
Clona el repositorio (cambia la URL por la de tu equipo). Si prefieres subir el zip, descomprímelo y haz `%cd` a la carpeta.

In [ ]:
REPO_URL = "https://github.com/USUARIO/pokemon-llm-benchmark.git"   # <-- cambiar
!git clone -q $REPO_URL repo 2>/dev/null || echo 'ya estaba clonado'
%cd repo
!pip install -q -r requirements-d2.txt
import os, sys
sys.path.insert(0, os.getcwd())

## 2. Tests de lógica (modelo falso, sin GPU)
Comprueban, entre otras cosas, que el generador reproduce exactamente los 30 estados del D1 y que **la solución nunca importa la tabla de tipos**.

In [ ]:
!python -m pytest -q tests

## 3. Cargar el modelo

In [ ]:
from d2.llm import HFModel
llm = HFModel("Qwen/Qwen2.5-1.5B-Instruct")   # T4 -> fp32 automatico (evita desbordes de fp16)

# chequeo rapido de que el modelo responde con sentido (no es parte de la evaluacion)
print(llm.generate("Say hello in one word.", max_new_tokens=5))

## 4. Evaluación: 50 estados × 7 estrategias + 50 estados nuevos
Si Colab se desconecta, vuelve a correr esta celda: retoma donde quedó.

In [ ]:
from d2.evaluate import run, HOLDOUT_SEED, HOLDOUT_STRATEGIES
from d2.report import build_report

# 50 estados x 7 estrategias (si results_d2/ ya existe, solo corre lo que falta)
run(llm, n=50, seed=20260830, out="results_d2")
print(build_report("results_d2"))

# 50 estados NUEVOS: baseline y solucion
run(llm, n=50, seed=HOLDOUT_SEED, out="results_d2_holdout", strategies=HOLDOUT_STRATEGIES)
print(build_report("results_d2_holdout"))

!python report/fill_d2.py results_d2 results_d2_holdout

## 5. Demo para el video: estado NUEVO
La semilla sale del reloj y se imprime, así que el estado no está elegido y cualquiera puede reproducirlo con `python -m d2.demo --seed <semilla>`.

In [ ]:
from d2.demo import run_demo
run_demo(llm)

## 6. Caso de fallo
Lista los estados donde la solución falla, con su causa, y repite el primero.

In [ ]:
import json
summary = json.load(open("results_d2/summary.json"))
for f in summary["failures"]:
    print(f["case_id"], f["difficulty"], "|", f["cause"], "|",
          [f"{w['attack']}->{w['defend']}: modelo {w['model']}, Gen I {w['truth']}" for w in f["wrong_facts"]])

from d2.demo import replay
if summary["failures"]:
    replay(llm, summary["failures"][0]["case_id"])

## 7. Descargar resultados
Sube `results_d2/` al repositorio: son los archivos de donde salen todas las cifras del documento.

In [ ]:
!zip -qr results_d2.zip results_d2 results_d2_holdout report/d2_numbers.tex
from google.colab import files
files.download("results_d2.zip")

## 8. (Opcional) Comparación con otro candidato del D1
Libera la GPU y corre baseline + solución con Phi-3.5-mini (3.8B) sobre los mismos 50 estados.

Sirve para ver si un modelo más grande **sabe** la tabla de tipos que Qwen2.5-1.5B no sabe (sección 5 de `summary.md`). No es parte de la entrega.

In [ ]:
import gc, torch
del llm; gc.collect(); torch.cuda.empty_cache()

phi = HFModel("microsoft/Phi-3.5-mini-instruct", dtype="float16")   # 3.8B en fp32 no cabe en la T4
run(phi, n=50, seed=20260830, out="results_d2_phi", strategies=["direct", "decomposed_words"])
print(build_report("results_d2_phi"))